In [2]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np

In [4]:
conn = sql.connect_pc()
cursor = conn.cursor()

query = """
    SELECT * FROM item_data
    WHERE type LIKE '%Misc'
    AND name NOT LIKE '%Gold Coin%'
    AND name NOT LIKE '%Volcano%'
    AND (name NOT LIKE 'Ruby%' and type LIKE '%Misc')
    AND name NOT LIKE '%Diamond%'
    AND name NOT LIKE '%Saphhire%'
    AND name NOT LIKE '%Emerald%';
    """

cursor.execute(query)
sql_fetch = cursor.fetchall()

In [9]:
from_date, to_date = "2026-01-25", "2026-04-29"
def get_url(item_id, from_data):
    return f"""https://api.darkerdb.com/v1/market?item_id={item_id}&limit=50&to={to_date}"""
url = get_url('Ectoplasm', from_date)
fetch = requests.get(url).json()['body']

rmt = [('name', 'rarity', 'amount', 'price', 'price_per_unit', 'created', 'sold')]
price_list = np.array([i['price_per_unit'] for i in fetch])
filter_method = np.percentile(price_list, 10) * 4
rmt_price = price_list[price_list > filter_method]
rmt.extend((i['item'], i['rarity'], i['quantity'], i['price'], i['price_per_unit'], i['created_at'], i['sold_at']) for i in fetch if i['price_per_unit'] > filter_method)

print(filter_method)
print(price_list.tolist())
print(rmt_price.tolist())


48552.0
[20000.0, 82255.0, 76539.0, 50000.0, 84444.0, 85000.0, 18888.0, 10.0, 77777.0, 20000.0, 87555.0, 88888.0, 100000.0, 100000.0, 19119.0, 18000.0, 19500.0, 19500.0, 20000.0, 29000.0, 12000.0, 12345.0, 24000.0, 9999.67, 28888.0]
[82255.0, 76539.0, 50000.0, 84444.0, 85000.0, 77777.0, 87555.0, 88888.0, 100000.0, 100000.0]


In [156]:
rmt[1:] = sorted(rmt[1:], key=lambda x: (x[3]), reverse=True)
for row in rmt:
    print(f"{row[0]:<30} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>20} {row[5]:>25} {str(row[6]):>25}")

name                               rarity     amount      price       price_per_unit                   created                      sold
Ectoplasm                        Uncommon          1     100000               100000      2026-04-28T04:42:27Z                      None
Ectoplasm                        Uncommon          1     100000               100000      2026-04-28T03:51:32Z      2026-04-28T04:09:23Z
Ectoplasm                        Uncommon          1      88888                88888      2026-04-28T04:43:56Z                      None
Ectoplasm                        Uncommon          1      87555                87555      2026-04-28T04:47:16Z      2026-04-28T05:36:43Z
Ectoplasm                        Uncommon          1      85000                85000      2026-04-28T06:58:47Z                      None
Ectoplasm                        Uncommon          1      84444                84444      2026-04-28T07:22:10Z                      None
Ectoplasm                        Uncommon

In [1]:
from_date = (datetime.now(timezone.utc) - timedelta(minutes=3)).strftime("%Y-%m-%dT%H:%M:%SZ")
from_date, to_date = "2026-04-01T18:00:00Z", "2026-04-21T19:00:00Z"

def get_url(item_id, from_data):
    return f"""https://api.darkerdb.com/v1/market?item_id={item_id}&limit=50&from={from_date}&to={to_date}"""

total, count = len(sql_fetch), 0

with requests.session() as ses:
    output = [('name', 'rarity', 'amount', 'price', 'price_per_unit', 'created', 'sold')]
    for item_id in sql_fetch:
        print(f"progress: {total}/{(count := count + 1)}", end="\r")
        fetch = ses.get(get_url(item_id[0], from_date)).json()
        output.extend((i['item'], i['rarity'], i['quantity'], i['price'], i['price_per_unit'], i['created_at'], i['sold_at']) for i in fetch['body'])

    

NameError: name 'datetime' is not defined

In [103]:
output[1:] = sorted(output[1:], key=lambda x: (x[3]), reverse=True)
for row in output:
    print(f"{row[0]:<30} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>15} {row[5]:>25} {str(row[6]):>25}")

name                               rarity     amount      price  price_per_unit                   created                      sold
Arcane Essence                  Legendary          1     100000          100000      2026-04-28T06:53:07Z      2026-04-28T07:15:22Z
Arcane Essence                     Unique          1     100000          100000      2026-04-28T08:35:07Z                      None
Arcane Essence                     Unique          1     100000          100000      2026-04-28T08:35:01Z                      None
Arcane Essence                     Unique          3     100000        33333.33      2026-04-28T08:34:58Z                      None
Arcane Essence                     Unique          1     100000          100000      2026-04-28T08:34:52Z                      None
Arcane Essence                     Unique          1     100000          100000      2026-04-28T08:34:41Z                      None
Ball                               Common          3     100000        33333